# Toyota Stock Market Intelligence (1980-2026) - EDA

Exploratory Data Analysis of Toyota stock data spanning over 45 years, including price data, volume, technical indicators (RSI, MACD, Moving Averages), and volatility measures.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

## 1. Data Loading & Overview

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/mdmahfuzsumon/toyota-stock-market-intelligence-19802026/toyota_stock_v2_features.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head()

In [ ]:
df.describe().round(3)

In [ ]:
df.info()

## 2. Price History

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})

# Price with moving averages
axes[0].plot(df['Date'], df['Close'], label='Close', alpha=0.8, linewidth=0.8)
axes[0].plot(df['Date'], df['MA_10'], label='MA 10', alpha=0.7, linewidth=0.8)
axes[0].plot(df['Date'], df['MA_50'], label='MA 50', alpha=0.7, linewidth=0.8)
axes[0].set_title('Toyota Stock Price with Moving Averages (1980-2026)')
axes[0].set_ylabel('Price (USD)')
axes[0].legend()

# Volume
axes[1].bar(df['Date'], df['Volume'], width=2, alpha=0.6, color='steelblue')
axes[1].set_title('Trading Volume')
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

## 3. Returns Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution of daily returns
axes[0].hist(df['Return'], bins=100, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].axvline(df['Return'].mean(), color='red', linestyle='--', label=f'Mean: {df["Return"].mean():.4f}')
axes[0].axvline(df['Return'].median(), color='orange', linestyle='--', label=f'Median: {df["Return"].median():.4f}')
axes[0].set_title('Distribution of Daily Returns')
axes[0].set_xlabel('Return')
axes[0].legend()

# QQ plot
from scipy import stats
stats.probplot(df['Return'].dropna(), dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Returns')

# Box plot by decade
df['Decade'] = (df['Date'].dt.year // 10) * 10
df.boxplot(column='Return', by='Decade', ax=axes[2])
axes[2].set_title('Returns by Decade')
axes[2].set_xlabel('Decade')
axes[2].set_ylabel('Return')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f'Skewness: {df["Return"].skew():.4f}')
print(f'Kurtosis: {df["Return"].kurtosis():.4f}')

In [ ]:
# Cumulative returns
df['Cumulative_Return'] = (1 + df['Return']).cumprod()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['Date'], df['Cumulative_Return'], color='steelblue')
ax.set_title('Cumulative Return Over Time')
ax.set_ylabel('Cumulative Return (x)')
ax.set_xlabel('Date')
plt.tight_layout()
plt.show()

print(f'Total cumulative return: {df["Cumulative_Return"].iloc[-1]:.2f}x')

## 4. Volatility Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df['Date'], df['Volatility_10'], label='10-day Volatility', alpha=0.8)
axes[0].plot(df['Date'], df['Volatility_30'], label='30-day Volatility', alpha=0.8)
axes[0].set_title('Rolling Volatility Over Time')
axes[0].set_ylabel('Volatility')
axes[0].legend()

# Volatility regime identification
vol_75 = df['Volatility_30'].quantile(0.75)
vol_90 = df['Volatility_30'].quantile(0.90)
axes[0].axhline(vol_75, color='orange', linestyle='--', alpha=0.5, label=f'75th pctl: {vol_75:.4f}')
axes[0].axhline(vol_90, color='red', linestyle='--', alpha=0.5, label=f'90th pctl: {vol_90:.4f}')
axes[0].legend()

# Scatter: Volatility vs absolute return
axes[1].scatter(df['Volatility_30'], df['Return'].abs(), alpha=0.1, s=5)
axes[1].set_title('30-day Volatility vs Absolute Daily Return')
axes[1].set_xlabel('Volatility (30-day)')
axes[1].set_ylabel('|Daily Return|')

plt.tight_layout()
plt.show()

## 5. Technical Indicators

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [2, 1, 1]})

# Price
recent = df[df['Date'] >= '2020-01-01'].copy()
axes[0].plot(recent['Date'], recent['Close'], label='Close', linewidth=1)
axes[0].plot(recent['Date'], recent['MA_10'], label='MA 10', alpha=0.7)
axes[0].plot(recent['Date'], recent['MA_50'], label='MA 50', alpha=0.7)
axes[0].set_title('Toyota Stock - Recent Period (2020+)')
axes[0].set_ylabel('Price')
axes[0].legend()

# RSI
axes[1].plot(recent['Date'], recent['RSI'], color='purple', linewidth=0.8)
axes[1].axhline(70, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(30, color='green', linestyle='--', alpha=0.5)
axes[1].fill_between(recent['Date'], 30, 70, alpha=0.1, color='gray')
axes[1].set_title('RSI (Relative Strength Index)')
axes[1].set_ylabel('RSI')
axes[1].set_ylim(0, 100)

# MACD
axes[2].plot(recent['Date'], recent['MACD'], label='MACD', linewidth=0.8)
axes[2].plot(recent['Date'], recent['MACD_Signal'], label='Signal', linewidth=0.8)
axes[2].bar(recent['Date'], recent['MACD'] - recent['MACD_Signal'], alpha=0.3, width=2, label='Histogram')
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_title('MACD')
axes[2].set_ylabel('MACD')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# RSI distribution and zones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['RSI'], bins=50, alpha=0.7, color='purple', edgecolor='white')
axes[0].axvline(30, color='green', linestyle='--', label='Oversold (30)')
axes[0].axvline(70, color='red', linestyle='--', label='Overbought (70)')
axes[0].set_title('RSI Distribution')
axes[0].set_xlabel('RSI')
axes[0].legend()

oversold_pct = (df['RSI'] < 30).mean() * 100
overbought_pct = (df['RSI'] > 70).mean() * 100
neutral_pct = 100 - oversold_pct - overbought_pct

axes[1].bar(['Oversold (<30)', 'Neutral (30-70)', 'Overbought (>70)'],
            [oversold_pct, neutral_pct, overbought_pct],
            color=['green', 'gray', 'red'], alpha=0.7)
axes[1].set_title('RSI Zone Distribution (%)')
axes[1].set_ylabel('Percentage of Days')

for i, v in enumerate([oversold_pct, neutral_pct, overbought_pct]):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
numeric_cols = ['Close', 'Volume', 'Return', 'MA_10', 'MA_50', 'Volatility_10', 'Volatility_30', 'RSI', 'MACD', 'MACD_Signal']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 7. Seasonal / Calendar Patterns

In [ ]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly average returns
monthly_returns = df.groupby('Month')['Return'].mean() * 100
colors = ['green' if x >= 0 else 'red' for x in monthly_returns]
axes[0].bar(monthly_returns.index, monthly_returns.values, color=colors, alpha=0.7)
axes[0].set_title('Average Daily Return by Month (%)')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Avg Return (%)')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)
axes[0].axhline(0, color='black', linewidth=0.5)

# Day of week returns
dow_returns = df.groupby('DayOfWeek')['Return'].mean() * 100
colors = ['green' if x >= 0 else 'red' for x in dow_returns]
axes[1].bar(dow_returns.index, dow_returns.values, color=colors, alpha=0.7)
axes[1].set_title('Average Daily Return by Day of Week (%)')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Avg Return (%)')
axes[1].set_xticks(range(5))
axes[1].set_xticklabels(['Mon','Tue','Wed','Thu','Fri'])
axes[1].axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Annual returns heatmap
annual_monthly = df.groupby(['Year', 'Month'])['Return'].sum().unstack()
annual_monthly = annual_monthly.loc[annual_monthly.index >= 1990]

fig, ax = plt.subplots(figsize=(14, 16))
sns.heatmap(annual_monthly * 100, cmap='RdYlGn', center=0, annot=True, fmt='.1f',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Monthly Return (%)'})
ax.set_title('Monthly Returns Heatmap (%) - Since 1990')
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_ylabel('Year')
plt.tight_layout()
plt.show()

## 8. Yearly Performance Summary

In [ ]:
yearly = df.groupby('Year').agg(
    Annual_Return=('Return', lambda x: (1 + x).prod() - 1),
    Avg_Volume=('Volume', 'mean'),
    Avg_Volatility=('Volatility_30', 'mean'),
    Max_Drawdown_Day=('Return', 'min'),
    Best_Day=('Return', 'max'),
    Close_YearEnd=('Close', 'last')
).round(4)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

colors = ['green' if x >= 0 else 'red' for x in yearly['Annual_Return']]
axes[0].bar(yearly.index, yearly['Annual_Return'] * 100, color=colors, alpha=0.7)
axes[0].set_title('Annual Returns (%)')
axes[0].set_ylabel('Return (%)')
axes[0].axhline(0, color='black', linewidth=0.5)

axes[1].plot(yearly.index, yearly['Avg_Volatility'], marker='o', markersize=3)
axes[1].set_title('Average Annual Volatility (30-day)')
axes[1].set_ylabel('Volatility')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.show()

print('Top 5 best years:')
print(yearly.nlargest(5, 'Annual_Return')[['Annual_Return', 'Avg_Volatility']].to_string())
print('\nTop 5 worst years:')
print(yearly.nsmallest(5, 'Annual_Return')[['Annual_Return', 'Avg_Volatility']].to_string())

## 9. Key Findings

Summary of the exploratory analysis:

- **Long-term trend**: Toyota stock shows significant long-term growth from ~$2 in 1980 to over $200 by 2026
- **Returns**: Daily returns are approximately normally distributed but with fat tails (excess kurtosis), typical of financial data
- **Volatility clustering**: Periods of high volatility tend to cluster together, visible especially during market crises
- **Technical indicators**: RSI and MACD provide useful signals; the stock spends most time in the neutral RSI zone (30-70)
- **Seasonality**: Calendar patterns may exist in monthly/weekly returns, worth further investigation for trading strategies
- **Correlations**: Strong correlation between price-related features (Close, MA_10, MA_50, MACD); volatility measures are highly correlated with each other